In [1]:
import json
import statistics
from pathlib import Path

results = Path('results/tcp-nodelay')

for enabled in [0, 1]:
    files = sorted(results.glob(f'run*-nodelay{enabled}.json'))

    data = [json.loads(path.read_text()) for path in files]

    p50 = [item['latency']['p50_us'] for item in data]
    p99 = [item['latency']['p99_us'] for item in data]
    throughput = [
        item['throughput']['orders_per_second']
        for item in data
    ]

    print(f'\nTCP_NODELAY={enabled}')
    print(f'Runs: {len(data)}')
    print(f'Median p50: {statistics.median(p50):.2f} us')
    print(f'Median p99: {statistics.median(p99):.2f} us')
    print(f'Median throughput: {statistics.median(throughput):.2f} orders/s')


TCP_NODELAY=0
Runs: 5
Median p50: 96.80 us
Median p99: 355.62 us
Median throughput: 3689.70 orders/s

TCP_NODELAY=1
Runs: 5
Median p50: 96.49 us
Median p99: 389.64 us
Median throughput: 50242.59 orders/s


In [2]:
import json
import statistics
from pathlib import Path

results = Path('results/tcp-batching')
summary = {}

for mode in [0, 1]:
    files = sorted(results.glob(f'run*-batch{mode}.json'))
    data = [json.loads(path.read_text()) for path in files]

    p50 = [item['latency']['p50_us'] for item in data]
    p99 = [item['latency']['p99_us'] for item in data]
    throughput = [
        item['throughput']['orders_per_second']
        for item in data
    ]

    summary[mode] = {
        'p50': statistics.median(p50),
        'p99': statistics.median(p99),
        'throughput': statistics.median(throughput)
    }

    print(f'\nWORKER_BATCH={mode}')
    print(f'Runs: {len(files)}')
    print(f'Median p50: {summary[mode]["p50"]:.2f} us')
    print(f'Median p99: {summary[mode]["p99"]:.2f} us')
    print(f'Median throughput: {summary[mode]["throughput"]:.2f} orders/s')

if len(summary) == 2:
    speedup = summary[1]['throughput'] / summary[0]['throughput']
    print(f'\nBatching throughput ratio: {speedup:.2f}x')


WORKER_BATCH=0
Runs: 5
Median p50: 123.52 us
Median p99: 646.54 us
Median throughput: 48577.79 orders/s

WORKER_BATCH=1
Runs: 5
Median p50: 127.08 us
Median p99: 671.05 us
Median throughput: 220524.26 orders/s

Batching throughput ratio: 4.54x


In [3]:

import json
import os
import statistics
import subprocess
from pathlib import Path

root = Path.home() / 'latency-lab'
results = root / 'results' / 'client-scaling'
results.mkdir(parents=True, exist_ok=True)

client_counts = [1, 2, 4, 8]
repetitions = 5
data = []

for run in range(1, repetitions + 1):
    clients_order = client_counts if run % 2 else client_counts[::-1]
    modes = [0, 1] if run % 2 else [1, 0]

    for clients in clients_order:
        for mode in modes:
            print(f'Run {run} | Clients={clients} | Batching={mode}', flush=True)

            env = os.environ.copy()
            env['LATENCY_TCP_NODELAY'] = '1'
            env['LATENCY_WORKER_BATCH'] = str(mode)

            command = [
                'python3',
                'benchmarks/tcp_benchmark.py',
                '--server', './build/latency_server',
                '--latency-samples', '100',
                '--warmup-pairs', '20',
                '--throughput-pairs', '5000',
                '--clients', str(clients),
                '--batch-pairs', '20'
            ]

            process = subprocess.run(
                command,
                cwd=root,
                env=env,
                capture_output=True,
                text=True,
                timeout=120
            )

            if process.returncode != 0:
                raise RuntimeError(
                    f'Benchmark failed:\n'
                    f'{process.stdout}\n{process.stderr}'
                )

            result = json.loads(process.stdout)

            assert result['throughput']['commands'] == 10000

            filename = f'clients{clients}-batch{mode}-run{run}.json'
            (results / filename).write_text(
                json.dumps(result, indent=2) + '\n'
            )

            data.append({
                'run': run,
                'clients': clients,
                'batching': mode,
                'throughput': result['throughput']['orders_per_second'],
                'p50_us': result['latency']['p50_us'],
                'p99_us': result['latency']['p99_us']
            })

print('\n=== CLIENT SCALING SUMMARY ===')

for clients in client_counts:
    print(f'\nClients: {clients}')
    medians = {}

    for mode in [0, 1]:
        values = [
            item['throughput']
            for item in data
            if item['clients'] == clients
            and item['batching'] == mode
        ]

        median = statistics.median(values)
        medians[mode] = median

        print(
            f'  Batching={mode}: '
            f'{median:,.2f} orders/s '
            f'(min={min(values):,.2f}, max={max(values):,.2f})'
        )

    speedup = medians[1] / medians[0]
    print(f'  Batching speedup: {speedup:.2f}x')


Run 1 | Clients=1 | Batching=0
Run 1 | Clients=1 | Batching=1
Run 1 | Clients=2 | Batching=0
Run 1 | Clients=2 | Batching=1
Run 1 | Clients=4 | Batching=0
Run 1 | Clients=4 | Batching=1
Run 1 | Clients=8 | Batching=0
Run 1 | Clients=8 | Batching=1
Run 2 | Clients=8 | Batching=1
Run 2 | Clients=8 | Batching=0
Run 2 | Clients=4 | Batching=1
Run 2 | Clients=4 | Batching=0
Run 2 | Clients=2 | Batching=1
Run 2 | Clients=2 | Batching=0
Run 2 | Clients=1 | Batching=1
Run 2 | Clients=1 | Batching=0
Run 3 | Clients=1 | Batching=0
Run 3 | Clients=1 | Batching=1
Run 3 | Clients=2 | Batching=0
Run 3 | Clients=2 | Batching=1
Run 3 | Clients=4 | Batching=0
Run 3 | Clients=4 | Batching=1
Run 3 | Clients=8 | Batching=0
Run 3 | Clients=8 | Batching=1
Run 4 | Clients=8 | Batching=1
Run 4 | Clients=8 | Batching=0
Run 4 | Clients=4 | Batching=1
Run 4 | Clients=4 | Batching=0
Run 4 | Clients=2 | Batching=1
Run 4 | Clients=2 | Batching=0
Run 4 | Clients=1 | Batching=1
Run 4 | Clients=1 | Batching=0
Run 5 | 

In [4]:

import json
import os
import statistics
import subprocess
from pathlib import Path

root = Path.home() / 'latency-lab'
results = root / 'results' / 'network-batch-scaling'
results.mkdir(parents=True, exist_ok=True)

batch_sizes = [1, 5, 10, 20, 40]
repetitions = 5
data = []

for run in range(1, repetitions + 1):
    sizes = batch_sizes if run % 2 else batch_sizes[::-1]
    modes = [0, 1] if run % 2 else [1, 0]

    for batch_size in sizes:
        for mode in modes:
            print(
                f'Run {run} | Network pairs={batch_size} | Worker batching={mode}',
                flush=True
            )

            env = os.environ.copy()
            env['LATENCY_TCP_NODELAY'] = '1'
            env['LATENCY_WORKER_BATCH'] = str(mode)

            command = [
                'python3',
                'benchmarks/tcp_benchmark.py',
                '--server', './build/latency_server',
                '--latency-samples', '100',
                '--warmup-pairs', '20',
                '--throughput-pairs', '5000',
                '--clients', '4',
                '--batch-pairs', str(batch_size)
            ]

            process = subprocess.run(
                command,
                cwd=root,
                env=env,
                capture_output=True,
                text=True,
                timeout=120
            )

            if process.returncode != 0:
                raise RuntimeError(
                    f'Benchmark failed:\n{process.stdout}\n{process.stderr}'
                )

            result = json.loads(process.stdout)
            assert result['throughput']['commands'] == 10000

            filename = f'pairs{batch_size}-worker{mode}-run{run}.json'
            (results / filename).write_text(
                json.dumps(result, indent=2) + '\n'
            )

            data.append({
                'run': run,
                'network_pairs': batch_size,
                'worker_batching': mode,
                'throughput': result['throughput']['orders_per_second']
            })

print('\n=== NETWORK BATCH SCALING SUMMARY ===')

for batch_size in batch_sizes:
    print(f'\nNetwork batch size: {batch_size} pairs')
    medians = {}

    for mode in [0, 1]:
        values = [
            item['throughput']
            for item in data
            if item['network_pairs'] == batch_size
            and item['worker_batching'] == mode
        ]

        median = statistics.median(values)
        medians[mode] = median

        print(
            f'  Worker batching={mode}: '
            f'{median:,.2f} orders/s '
            f'(min={min(values):,.2f}, max={max(values):,.2f})'
        )

    print(
        f'  Worker batching speedup: '
        f'{medians[1] / medians[0]:.2f}x'
    )


Run 1 | Network pairs=1 | Worker batching=0
Run 1 | Network pairs=1 | Worker batching=1
Run 1 | Network pairs=5 | Worker batching=0
Run 1 | Network pairs=5 | Worker batching=1
Run 1 | Network pairs=10 | Worker batching=0
Run 1 | Network pairs=10 | Worker batching=1
Run 1 | Network pairs=20 | Worker batching=0
Run 1 | Network pairs=20 | Worker batching=1
Run 1 | Network pairs=40 | Worker batching=0
Run 1 | Network pairs=40 | Worker batching=1
Run 2 | Network pairs=40 | Worker batching=1
Run 2 | Network pairs=40 | Worker batching=0
Run 2 | Network pairs=20 | Worker batching=1
Run 2 | Network pairs=20 | Worker batching=0
Run 2 | Network pairs=10 | Worker batching=1
Run 2 | Network pairs=10 | Worker batching=0
Run 2 | Network pairs=5 | Worker batching=1
Run 2 | Network pairs=5 | Worker batching=0
Run 2 | Network pairs=1 | Worker batching=1
Run 2 | Network pairs=1 | Worker batching=0
Run 3 | Network pairs=1 | Worker batching=0
Run 3 | Network pairs=1 | Worker batching=1
Run 3 | Network pair

In [5]:

import json
import os
import select
import socket
import statistics
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from threading import Barrier


def read_response(stream):
    lines = []

    while True:
        line = stream.readline()

        if not line:
            raise RuntimeError('Server disconnected before END')

        value = line.decode('utf-8').rstrip('\r\n')

        if value == 'END':
            return lines

        lines.append(value)


def client_job(client_id, port, count, sell_barrier, buy_barrier):
    matched_sells = []
    response_times = []

    with socket.create_connection(('127.0.0.1', port), timeout=10) as conn:
        conn.settimeout(30)

        with conn.makefile('rb') as stream:
            sell_barrier.wait(timeout=30)

            commands = ''.join(
                f'SELL s{client_id}_{i} 100 1\n'
                for i in range(count)
            )
            conn.sendall(commands.encode())

            for i in range(count):
                expected = [f'ACCEPTED s{client_id}_{i}, trades=0']
                assert read_response(stream) == expected

            buy_barrier.wait(timeout=30)

            commands = ''.join(
                f'BUY b{client_id}_{i} 100 1\n'
                for i in range(count)
            )
            conn.sendall(commands.encode())

            for i in range(count):
                response = read_response(stream)
                received_ns = time.perf_counter_ns()

                assert len(response) == 2, response
                assert response[0] == (
                    f'ACCEPTED b{client_id}_{i}, trades=1'
                ), response

                parts = response[1].split()

                assert len(parts) == 5, parts
                assert parts[0] == 'TRADE', parts
                assert parts[1] == f'buy=b{client_id}_{i}', parts
                assert parts[2].startswith('sell=s'), parts
                assert parts[3] == 'price=100', parts
                assert parts[4] == 'qty=1', parts

                matched_sells.append(parts[2].split('=', 1)[1])
                response_times.append(received_ns)

    return {
        'client_id': client_id,
        'matched_sells': matched_sells,
        'response_times': response_times
    }


def run_fairness(root, mode, clients=8, orders_per_client=1000):
    env = os.environ.copy()
    env['LATENCY_TCP_NODELAY'] = '1'
    env['LATENCY_WORKER_BATCH'] = str(mode)

    process = subprocess.Popen(
        [str(root / 'build' / 'latency_server'), '0'],
        cwd=root,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    try:
        readable, _, _ = select.select(
            [process.stdout], [], [], 10
        )

        if not readable:
            raise RuntimeError('Server startup timeout')

        line = process.stdout.readline().strip()

        if not line.startswith('LISTENING 127.0.0.1:'):
            raise RuntimeError(f'Unexpected server output: {line}')

        port = int(line.rsplit(':', 1)[1])

        sell_barrier = Barrier(clients)
        start_ns = [0]

        def mark_start():
            start_ns[0] = time.perf_counter_ns()

        buy_barrier = Barrier(clients, action=mark_start)

        with ThreadPoolExecutor(max_workers=clients) as executor:
            futures = [
                executor.submit(
                    client_job,
                    client_id,
                    port,
                    orders_per_client,
                    sell_barrier,
                    buy_barrier
                )
                for client_id in range(clients)
            ]

            outputs = [future.result() for future in futures]

        matched = [
            sell_id
            for item in outputs
            for sell_id in item['matched_sells']
        ]

        expected = {
            f's{client_id}_{i}'
            for client_id in range(clients)
            for i in range(orders_per_client)
        }

        assert len(matched) == clients * orders_per_client
        assert len(set(matched)) == len(matched)
        assert set(matched) == expected

        with socket.create_connection(
            ('127.0.0.1', port), timeout=10
        ) as conn:
            with conn.makefile('rb') as stream:
                conn.sendall(b'BOOK\n')
                assert read_response(stream) == ['BIDS', 'ASKS']

        completion_ms = [
            (item['response_times'][-1] - start_ns[0]) / 1e6
            for item in outputs
        ]

        last_ns = max(
            item['response_times'][-1]
            for item in outputs
        )

        midpoint_ns = start_ns[0] + (last_ns - start_ns[0]) // 2

        midpoint_progress = [
            sum(
                timestamp <= midpoint_ns
                for timestamp in item['response_times']
            )
            for item in outputs
        ]

        total_progress = sum(midpoint_progress)
        squared_progress = sum(
            value ** 2 for value in midpoint_progress
        )

        balance = (
            total_progress ** 2 / (clients * squared_progress)
            if squared_progress else 0.0
        )

        return {
            'worker_batching': mode,
            'clients': clients,
            'orders_per_client': orders_per_client,
            'verified_trades': len(matched),
            'completion_ms': [
                round(value, 3) for value in completion_ms
            ],
            'median_completion_ms': round(
                statistics.median(completion_ms), 3
            ),
            'max_completion_ms': round(max(completion_ms), 3),
            'min_completion_ms': round(min(completion_ms), 3),
            'completion_ratio': round(
                max(completion_ms) / min(completion_ms), 3
            ),
            'midpoint_progress': midpoint_progress,
            'midpoint_balance': round(balance, 4)
        }

    finally:
        process.terminate()

        try:
            process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.communicate()


root = Path.home() / 'latency-lab'
results = root / 'results' / 'fairness'
results.mkdir(parents=True, exist_ok=True)

all_results = []

for run in range(1, 6):
    modes = [0, 1] if run % 2 else [1, 0]

    for mode in modes:
        print(f'Run {run} | Worker batching={mode}', flush=True)

        result = run_fairness(root, mode)

        path = results / f'run{run}-batch{mode}.json'
        path.write_text(json.dumps(result, indent=2) + '\n')

        all_results.append(result)

        print(
            f'  Trades={result["verified_trades"]}, '
            f'Median completion={result["median_completion_ms"]:.3f} ms, '
            f'Balance={result["midpoint_balance"]:.4f}'
        )


print('\n=== FAIRNESS SUMMARY ===')

for mode in [0, 1]:
    items = [
        item for item in all_results
        if item['worker_batching'] == mode
    ]

    print(f'\nWorker batching={mode}')
    print(f'Runs: {len(items)}')

    print(
        'Median client completion: '
        f'{statistics.median(item["median_completion_ms"] for item in items):.3f} ms'
    )

    print(
        'Median slowest client completion: '
        f'{statistics.median(item["max_completion_ms"] for item in items):.3f} ms'
    )

    print(
        'Median max/min completion ratio: '
        f'{statistics.median(item["completion_ratio"] for item in items):.3f}x'
    )

    print(
        'Median midpoint progress balance: '
        f'{statistics.median(item["midpoint_balance"] for item in items):.4f}'
    )

    print(
        'Verified trades: '
        f'{sum(item["verified_trades"] for item in items):,}'
    )


Run 1 | Worker batching=0
  Trades=8000, Median completion=54.176 ms, Balance=0.9908
Run 1 | Worker batching=1
  Trades=8000, Median completion=21.514 ms, Balance=0.8621
Run 2 | Worker batching=1
  Trades=8000, Median completion=23.304 ms, Balance=0.8678
Run 2 | Worker batching=0
  Trades=8000, Median completion=48.334 ms, Balance=0.9976
Run 3 | Worker batching=0
  Trades=8000, Median completion=73.182 ms, Balance=0.9713
Run 3 | Worker batching=1
  Trades=8000, Median completion=33.404 ms, Balance=0.8972
Run 4 | Worker batching=1
  Trades=8000, Median completion=27.107 ms, Balance=0.8731
Run 4 | Worker batching=0
  Trades=8000, Median completion=47.595 ms, Balance=0.9968
Run 5 | Worker batching=0
  Trades=8000, Median completion=51.056 ms, Balance=0.9954
Run 5 | Worker batching=1
  Trades=8000, Median completion=20.009 ms, Balance=0.8669

=== FAIRNESS SUMMARY ===

Worker batching=0
Runs: 5
Median client completion: 51.056 ms
Median slowest client completion: 51.938 ms
Median max/min co